# Auto Loader Ingestion – Car Workshop Franchise

Incrementally ingests parquet files from Unity Catalog Volumes into Delta tables.

| Widget | Values | Description |
|--------|--------|-------------|
| `TRIGGER_MODE` | `availableNow` / `continuous` | **availableNow** – process all files then stop (use for initial load & Auto Loader testing with single-day data). **continuous** – keep stream running, picks up new files every 30 s. |
| `SINGLE_TABLE` | table name or blank | Leave blank to ingest all 20 tables. Set e.g. `dim_locations` to run one table only. |

**Volume layout:**
```
/Volumes/fake_car_workshop_franchise/
  dim/
    dim_parquet_files/       ← source (generator output)
    autoloader_checkpoints/  ← checkpoints + schema locations (this notebook)
  fact/
    fact_parquet_files/      ← source
    autoloader_checkpoints/  ← checkpoints + schema locations
```

In [0]:
import os
from pyspark.sql.functions import col

print("Imports OK")

In [0]:
dbutils.widgets.dropdown(
    'TRIGGER_MODE', 'availableNow', ['availableNow', 'continuous'],
    label='Trigger Mode'
)
dbutils.widgets.text(
    'SINGLE_TABLE', '',
    label='Single Table  (blank = all, e.g. dim_locations)'
)

TRIGGER_MODE = dbutils.widgets.get('TRIGGER_MODE')
SINGLE_TABLE = dbutils.widgets.get('SINGLE_TABLE').strip()

print(f'TRIGGER_MODE = {TRIGGER_MODE}')
print(f'SINGLE_TABLE = {SINGLE_TABLE or "(all tables)"}')

In [0]:
CATALOG              = 'fake_car_workshop_franchise_pl'

DIM_PARQUET_BASE     = f'/Volumes/{CATALOG}/dim/dim_daily_files'
FACT_PARQUET_BASE    = f'/Volumes/{CATALOG}/fact/fact_daily_files'
DIM_CHECKPOINT_BASE  = f'/Volumes/{CATALOG}/dim/autoloader_checkpoints'
FACT_CHECKPOINT_BASE = f'/Volumes/{CATALOG}/fact/autoloader_checkpoints'

print(f'Source  DIM :  {DIM_PARQUET_BASE}')
print(f'Source  FACT:  {FACT_PARQUET_BASE}')
print(f'Chkpts  DIM :  {DIM_CHECKPOINT_BASE}')
print(f'Chkpts  FACT:  {FACT_CHECKPOINT_BASE}')

In [0]:
def schema_to_ddl(d):
    return ", ".join(f"`{c}` {t}" for c, t in d.items())


TABLE_SCHEMAS = {
    # ── dimensions ──────────────────────────────────────────────────────
    
    "dim_locations": {
        "location_id": "BIGINT",
        "location_code": "STRING",
        "nazwa": "STRING",
        "typ": "STRING",
        "ulica": "STRING",
        "miasto": "STRING",
        "wojewodztwo": "STRING",
        "kod_pocztowy": "STRING",
        "latitude": "DOUBLE",
        "longitude": "DOUBLE",
        "telefon": "STRING",
        "email": "STRING",
        "kierownik_id": "BIGINT",
        "liczba_stanowisk": "BIGINT",
        "powierzchnia_m2": "BIGINT",
        "data_otwarcia": "DATE",
        "czy_aktywna": "BOOLEAN",
    },
    "dim_employees": {
        # lista nad tabelą była już po angielsku — bez zmian
        "employee_id": "BIGINT",
        "employee_code": "STRING",
        "first_name": "STRING",
        "last_name": "STRING",
        "national_id": "STRING",
        "position": "STRING",
        "location_id": "BIGINT",
        "hire_date": "DATE",
        "termination_date": "DATE",
        "hourly_rate": "DOUBLE",
        "is_active": "BOOLEAN",
    },
    "dim_customers": {
        "customer_id": "BIGINT",
        "customer_code": "STRING",
        "typ_klienta": "STRING",
        "imie": "STRING",
        "nazwisko": "STRING",
        "nazwa_firmy": "STRING",
        "nip": "STRING",
        "email": "STRING",
        "telefon": "STRING",
        "miasto": "STRING",
        "kod_pocztowy": "STRING",
        "data_rejestracji": "timestamp_ntz",
        "preferowana_lokalizacja_id": "BIGINT",
        "zgoda_marketing": "BOOLEAN",
    },
    "dim_vehicles": {
        "vehicle_id": "BIGINT",
        "customer_id": "BIGINT",
        "marka": "STRING",
        "model": "STRING",
        "rocznik": "BIGINT",
        "vin": "STRING",
        "nr_rejestracyjny": "STRING",
        "typ_paliwa": "STRING",
        "pojemnosc_silnika": "DOUBLE",
        "moc_km": "BIGINT",
        "kolor": "STRING",
        "przebieg_km": "BIGINT",
        "data_pierwszej_rejestracji": "timestamp_ntz",
    },
    "dim_products": {
        "product_id": "BIGINT",
        "product_code": "STRING",
        "nazwa": "STRING",
        "kategoria": "STRING",
        "producent": "STRING",
        "cena_zakupu_netto": "DOUBLE",
        "cena_sprzedazy_netto": "DOUBLE",
        "vat_procent": "BIGINT",
        "jednostka": "STRING",
        "waga_kg": "DOUBLE",
        "min_stan_magazynowy": "BIGINT",
        "czy_aktywny": "BOOLEAN",
    },
    "dim_services": {
        "service_id": "BIGINT",
        "service_code": "STRING",
        "nazwa": "STRING",
        "kategoria": "STRING",
        "cena_min_netto": "BIGINT",
        "cena_max_netto": "BIGINT",
        "szacowany_czas_min": "BIGINT",
        "czy_aktywna": "BOOLEAN",
    },
    "dim_suppliers": {
        "supplier_id": "BIGINT",
        "supplier_code": "STRING",
        "nazwa": "STRING",
        "nip": "STRING",
        "miasto": "STRING",
        "adres": "STRING",
        "kod_pocztowy": "STRING",
        "telefon": "STRING",
        "email": "STRING",
        "osoba_kontaktowa": "STRING",
        "warunki_platnosci_dni": "BIGINT",
        "min_wartosc_zamowienia": "DOUBLE",
        "czy_aktywny": "BOOLEAN",
    },
    # ── facts ────────────────────────────────────────────────────────────
    "fact_work_orders": {
        "work_order_id": "BIGINT",
        "work_order_code": "STRING",
        "location_id": "bigint",
        "customer_id": "bigint",
        "vehicle_id": "bigint",
        "mechanic_id": "bigint",
        "data_przyjecia": "timestamp_ntz",
        "data_zakonczenia": "timestamp_ntz",
        "status": "STRING",
        "przebieg_przy_przyjeciu": "bigint",
        "uwagi_klienta": "STRING",
        "rok": "INT",
        "miesiac": "INT",
    },
    "fact_work_order_items": {
        "wo_item_id": "BIGINT",
        "work_order_id": "BIGINT",
        "typ_pozycji": "STRING",
        "service_id": "bigint",
        "product_id": "bigint",
        "ilosc": "bigint",
        "cena_jednostkowa_netto": "DOUBLE",
        "wartosc_netto": "DOUBLE",
        "vat_procent": "bigint",
        "wartosc_brutto": "DOUBLE",
        "rabat_procent": "bigint",
    },
    "fact_sales_transactions": {
        "transaction_id": "BIGINT",
        "transaction_code": "STRING",
        "location_id": "bigint",
        "customer_id": "bigint",
        "employee_id": "bigint",
        "data_transakcji": "timestamp_ntz",
        "metoda_platnosci": "STRING",
        "nr_paragonu": "STRING",
        "rok": "int",
        "miesiac": "int",
    },
    "fact_sales_items": {
        "sales_item_id": "BIGINT",
        "transaction_id": "BIGINT",
        "product_id": "bigint",
        "ilosc": "bigint",
        "cena_jednostkowa_netto": "DOUBLE",
        "rabat_procent": "bigint",
        "wartosc_netto": "DOUBLE",
        "vat_procent": "bigint",
        "wartosc_brutto": "DOUBLE",
    },
    "fact_invoices": {
        "invoice_id": "BIGINT",
        "invoice_code": "STRING",
        "typ_dokumentu": "STRING",
        "source_type": "STRING",
        "source_id": "BIGINT",
        "customer_id": "bigint",
        "location_id": "bigint",
        "data_wystawienia": "timestamp_ntz",
        "data_sprzedazy": "timestamp_ntz",
        "termin_platnosci": "timestamp_ntz",
        "wartosc_netto": "DOUBLE",
        "wartosc_vat": "DOUBLE",
        "wartosc_brutto": "DOUBLE",
        "status": "STRING",
        "rok": "INT",
        "miesiac": "INT",
    },
    "fact_payments": {
        "payment_id": "BIGINT",
        "invoice_id": "BIGINT",
        "data_platnosci": "timestamp_ntz",
        "kwota": "DOUBLE",
        "metoda_platnosci": "STRING",
        "status": "STRING",
        "numer_transakcji": "STRING",
        "rok": "INT",
        "miesiac": "INT",
    },
    "fact_inventory_movements": {
        "movement_id": "BIGINT",
        "product_id": "bigint",
        "location_id": "bigint",
        "typ_ruchu": "STRING",
        "ilosc": "bigint",
        "data_ruchu": "timestamp_ntz",
        "dokument_zrodlowy": "STRING",
        "nr_dokumentu": "STRING",
        "wartosc_netto": "DOUBLE",
        "uwagi": "STRING",
        "rok": "INT",
        "miesiac": "INT",
    },
    "fact_appointments": {
        "appointment_id": "BIGINT",
        "customer_id": "bigint",
        "vehicle_id": "bigint",
        "location_id": "bigint",
        "service_id": "bigint",
        "data_rezerwacji": "timestamp_ntz",
        "data_wizyty": "TIMESTAMP",
        "status": "STRING",
        "kanal_rezerwacji": "STRING",
        "uwagi": "STRING",
        "rok": "INT",
        "miesiac": "INT",
    },
    "fact_purchase_orders": {
        "po_id": "BIGINT",
        "po_code": "STRING",
        "supplier_id": "bigint",
        "location_id": "bigint",
        "data_zamowienia": "timestamp_ntz",
        "data_dostawy_planowana": "timestamp_ntz",
        "data_dostawy_rzeczywista": "timestamp_ntz",
        "wartosc_netto": "DOUBLE",
        "wartosc_brutto": "DOUBLE",
        "status": "STRING",
        "rok": "INT",
    },
    "fact_purchase_order_items": {
        "po_item_id": "BIGINT",
        "po_id": "BIGINT",
        "product_id": "bigint",
        "ilosc_zamowiona": "bigint",
        "ilosc_dostarczona": "bigint",
        "cena_jednostkowa_netto": "DOUBLE",
        "wartosc_netto": "DOUBLE",
    },
    "fact_customer_feedback": {
        "feedback_id": "BIGINT",
        "customer_id": "bigint",
        "location_id": "bigint",
        "work_order_id": "BIGINT",
        "data_opinii": "timestamp_ntz",
        "ocena": "bigint",
        "komentarz": "STRING",
        "kategoria": "STRING",
        "kanal": "STRING",
    },
    "fact_loyalty_program": {
        "loyalty_id": "BIGINT",
        "customer_id": "bigint",
        "data_zdarzenia": "timestamp_ntz",
        "typ_zdarzenia": "STRING",
        "punkty": "bigint",
        "opis": "STRING",
        "saldo_po": "bigint",
        "poziom": "STRING",
    },
    "fact_employee_schedules": {
        "schedule_id": "BIGINT",
        "employee_id": "bigint",
        "data": "timestamp_ntz",
        "godzina_start": "bigint",
        "godzina_koniec": "bigint",
        "typ_zmiany": "STRING",
        "nadgodziny_h": "bigint",
        "attendance": "STRING",
    },
}

# fact tables that carry partition columns in their parquet directory structure
PARTITIONED_TABLES = {
    "fact_work_orders": ["rok", "miesiac"],
    "fact_sales_transactions": ["rok", "miesiac"],
    "fact_invoices": ["rok", "miesiac"],
    "fact_payments": ["rok", "miesiac"],
    "fact_inventory_movements": ["rok", "miesiac"],
    "fact_appointments": ["rok", "miesiac"],
    "fact_purchase_orders": ["rok"],
}

print(f"Schemas loaded: {len(TABLE_SCHEMAS)} tables")

In [0]:
# Keeps references to running streams when TRIGGER_MODE = continuous
active_streams = []


def ingest_table(table_name, schema_name, trigger_mode='availableNow'):
    is_dim          = schema_name == 'dim'
    source_base     = DIM_PARQUET_BASE     if is_dim else FACT_PARQUET_BASE
    checkpoint_base = DIM_CHECKPOINT_BASE  if is_dim else FACT_CHECKPOINT_BASE

    source_path     = f'{source_base}/{table_name}'
    checkpoint_path = f'{checkpoint_base}/{table_name}/checkpoint'
    schema_location = f'{checkpoint_base}/{table_name}/schema'
    target_table    = f'{CATALOG}.{schema_name}.{table_name}'
    partition_cols  = PARTITIONED_TABLES.get(table_name)

    print(f'  {table_name}  ->  {target_table}')

    reader = (
        spark.readStream
            .format('cloudFiles')
            .option('cloudFiles.format', 'parquet')
            .option('cloudFiles.schemaLocation', schema_location)
            .option('cloudFiles.inferColumnTypes', 'false')   # use our explicit schema
            .schema(schema_to_ddl(TABLE_SCHEMAS[table_name]))
            .load(source_path)
    )

    writer = (
        reader.writeStream
            .format('delta')
            .outputMode('append')
            .option('checkpointLocation', checkpoint_path)
            .option('mergeSchema', 'true')
    )

    if partition_cols:
        writer = writer.partitionBy(*partition_cols)

    if trigger_mode == 'availableNow':
        # Process all currently available files, then stop automatically
        query = writer.trigger(availableNow=True).toTable(target_table)
        query.awaitTermination()
        print(f'     done  ({query.lastProgress["numInputRows"] if query.lastProgress else "?"} rows in last micro-batch)')
    else:
        # Keep stream alive; picks up new files every 30 s
        query = writer.trigger(processingTime='30 seconds').toTable(target_table)
        active_streams.append((table_name, query))
        print(f'     stream started  (id={query.id})')

    return query


print('ingest_table() ready')

## Dimension Tables

7 tables – no partitioning.

In [0]:
DIM_TABLES = [
    'dim_locations',
    'dim_employees',
    'dim_customers',
    'dim_vehicles',
    'dim_products',
    'dim_services',
    'dim_suppliers',
]

print('=== Dimension tables ===')
for table in DIM_TABLES:
    if not SINGLE_TABLE or SINGLE_TABLE == table:
        ingest_table(table, 'dim', TRIGGER_MODE)
print('Done.')

## Fact Tables

13 tables – 7 are partitioned by `year` / `month` in the parquet layout.

In [0]:
FACT_TABLES = [
    'fact_work_orders',           # partitioned year/month
    'fact_work_order_items',
    'fact_sales_transactions',    # partitioned year/month
    'fact_sales_items',
    'fact_invoices',              # partitioned year/month
    'fact_payments',              # partitioned year/month
    'fact_inventory_movements',   # partitioned year/month
    'fact_appointments',          # partitioned year/month
    'fact_purchase_orders',       # partitioned year
    'fact_purchase_order_items',
    'fact_customer_feedback',
    'fact_loyalty_program',
    'fact_employee_schedules',
]

print('=== Fact tables ===')
for table in FACT_TABLES:
    if not SINGLE_TABLE or SINGLE_TABLE == table:
        ingest_table(table, 'fact', TRIGGER_MODE)
print('Done.')

## Stop All Continuous Streams

Run the cell below to gracefully stop every active stream  
(only relevant when `TRIGGER_MODE = continuous`).

In [0]:
if not active_streams:
    print('No active streams to stop.')
else:
    for table_name, query in active_streams:
        query.stop()
        print(f'Stopped: {table_name}')
    active_streams.clear()
    print('All streams stopped.')

## Validation

Row counts for every ingested table.

In [0]:
all_tables = [('dim', t) for t in DIM_TABLES] + [('fact', t) for t in FACT_TABLES]

results = []
for schema_name, table_name in all_tables:
    full_name = f'{CATALOG}.{schema_name}.{table_name}'
    try:
        count = spark.table(full_name).count()
        results.append({'table': full_name, 'row_count': count, 'status': 'OK'})
    except Exception as e:
        results.append({'table': full_name, 'row_count': None, 'status': str(e)[:60]})

import pandas as pd
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))